<a href="https://colab.research.google.com/github/Sarah1542/Smart-Adaptive-Environment-Monitoring-Decision-System/blob/GNN/GNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Installs

In [ ]:
# ==============================
# 1. INSTALL PACKAGES
# ==============================
!pip install torch
!pip install torch-scatter torch-sparse torch-cluster torch-spline-conv
!pip install torch-geometric



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.0/108.0 kB 7.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.0/210.0 kB 17.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 5.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
ERROR: Operation cancelled by user
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 423, in run
    _, build_failures = build(
                        ^^^^^^
  File "/

#Imports and Graph Data Preparation

In [ ]:

# ==============================
# 2. IMPORTS
# ==============================
import json
import torch
import networkx as nx
from torch_geometric.data import Data


# ==============================
# 3. LOAD JSON FILE
# ==============================
with open("topology_graph.json", "r") as f:
    data = json.load(f)

nodes = data["nodes"]
edges = data["edges"]


# ==============================
# 4. NETWORKX GRAPH (OPTIONAL)
# ==============================
G = nx.Graph()

for node in nodes:
    G.add_node(node["id"], name=node["name"], type=node["type"])

for edge in edges:
    G.add_edge(edge["source"], edge["target"], distance=edge["distance_meters"])


# ==============================
# 5. EDGE INDEX + EDGE ATTR
# ==============================
edge_index = []
edge_attr = []

for edge in edges:
    s, t = edge["source"], edge["target"]
    d = edge["distance_meters"]

    # undirected graph
    edge_index.append([s, t])
    edge_index.append([t, s])

    edge_attr.append([d])
    edge_attr.append([d])

edge_index = torch.tensor(edge_index).t().long()
edge_attr = torch.tensor(edge_attr).float()


# ==============================
# 6. NODE FEATURES (ONE-HOT TYPE)
# ==============================
types = list(set(node["type"] for node in nodes))
type_to_idx = {t: i for i, t in enumerate(types)}

x = []

for node in nodes:
    one_hot = [0] * len(types)
    one_hot[type_to_idx[node["type"]]] = 1
    x.append(one_hot)

x = torch.tensor(x).float()


# ==============================
# 7. FINAL GRAPH DATA
# ==============================
graph_data = Data(
    x=x,
    edge_index=edge_index,
    edge_attr=edge_attr
)

print(graph_data)

Data(x=[4, 4], edge_index=[2, 8], edge_attr=[8, 1])


#GNN Model

In [ ]:
# ==============================
# FULL PURE PYTORCH GNN FROM JSON
# ==============================

import json
import torch
import torch.nn as nn
import torch.nn.functional as F

# ------------------------------
# LOAD JSON FILE
# ------------------------------
with open("topology_graph.json", "r") as f:
    data = json.load(f)

nodes = data["nodes"]
edges = data["edges"]


# ------------------------------
# DEVICE
# ------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ------------------------------
# NODE FEATURES (ONE-HOT ENCODING)
# ------------------------------
types = list(set(node["type"] for node in nodes))
type_to_idx = {t: i for i, t in enumerate(types)}

x = []

for node in nodes:
    one_hot = [0] * len(types)
    one_hot[type_to_idx[node["type"]]] = 1
    x.append(one_hot)

x = torch.tensor(x, dtype=torch.float32).to(device)


# ------------------------------
# LABELS
# ------------------------------
type_to_label = {
    "temperature": 0,
    "humidity": 1,
    "gas": 2,
    "light": 3
}

y = torch.tensor(
    [type_to_label[node["type"]] for node in nodes],
    dtype=torch.long
).to(device)


# ------------------------------
# BUILD ADJACENCY MATRIX
# ------------------------------
num_nodes = len(nodes)

adj = torch.zeros((num_nodes, num_nodes), dtype=torch.float32)

for edge in edges:
    s = edge["source"]
    t = edge["target"]

    adj[s, t] = 1
    adj[t, s] = 1   # undirected graph

# Add self-loops
adj += torch.eye(num_nodes)

# Normalize adjacency matrix
deg = adj.sum(dim=1)
deg_inv_sqrt = torch.diag(torch.pow(deg, -0.5))

adj = deg_inv_sqrt @ adj @ deg_inv_sqrt
adj = adj.to(device)


# ------------------------------
# CUSTOM GCN LAYER
# ------------------------------
class GCNLayer(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features)

    def forward(self, x, adj):
        x = adj @ x
        x = self.linear(x)
        return x


# ------------------------------
# FULL GNN MODEL
# ------------------------------
class PureGNN(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()

        self.gcn1 = GCNLayer(in_channels, hidden_channels)
        self.gcn2 = GCNLayer(hidden_channels, hidden_channels)
        self.gcn3 = GCNLayer(hidden_channels, out_channels)

        self.dropout = nn.Dropout(0.3)

    def forward(self, x, adj):
        x = self.gcn1(x, adj)
        x = F.relu(x)
        x = self.dropout(x)

        x = self.gcn2(x, adj)
        x = F.relu(x)
        x = self.dropout(x)

        x = self.gcn3(x, adj)

        return x


# ------------------------------
# MODEL INIT
# ------------------------------
model = PureGNN(
    in_channels=x.shape[1],
    hidden_channels=32,
    out_channels=4
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01,
    weight_decay=1e-4
)

loss_fn = nn.CrossEntropyLoss()


# ------------------------------
# TRAINING LOOP
# ------------------------------
for epoch in range(300):
    model.train()

    optimizer.zero_grad()

    out = model(x, adj)

    loss = loss_fn(out, y)

    loss.backward()

    optimizer.step()

    if epoch % 20 == 0:
        pred = out.argmax(dim=1)
        acc = (pred == y).float().mean()

        print(
            f"Epoch {epoch} | "
            f"Loss: {loss.item():.4f} | "
            f"Accuracy: {acc:.4f}"
        )


# ------------------------------
# FINAL EVALUATION
# ------------------------------
model.eval()

with torch.no_grad():
    out = model(x, adj)
    pred = out.argmax(dim=1)

accuracy = (pred == y).float().mean()

print("\nFinal Predictions:", pred.cpu().numpy())
print("True Labels:", y.cpu().numpy())
print("Final Accuracy:", accuracy.item())

Epoch 0 | Loss: 1.4056 | Accuracy: 0.2500
Epoch 20 | Loss: 1.2812 | Accuracy: 0.5000
Epoch 40 | Loss: 0.8255 | Accuracy: 0.7500
Epoch 60 | Loss: 0.7398 | Accuracy: 0.5000
Epoch 80 | Loss: 0.6254 | Accuracy: 0.7500
Epoch 100 | Loss: 0.4496 | Accuracy: 0.7500
Epoch 120 | Loss: 0.4304 | Accuracy: 0.7500
Epoch 140 | Loss: 0.6042 | Accuracy: 0.7500
Epoch 160 | Loss: 0.4751 | Accuracy: 0.7500
Epoch 180 | Loss: 0.4089 | Accuracy: 0.7500
Epoch 200 | Loss: 0.4433 | Accuracy: 0.7500
Epoch 220 | Loss: 0.7223 | Accuracy: 0.5000
Epoch 240 | Loss: 0.4710 | Accuracy: 0.7500
Epoch 260 | Loss: 0.4565 | Accuracy: 0.7500
Epoch 280 | Loss: 0.3589 | Accuracy: 0.7500

Final Predictions: [0 0 2 3]
True Labels: [0 1 2 3]
Final Accuracy: 0.75


#Model saving

In [ ]:
# ==============================
# SAVE TRAINED MODEL + RESULTS
# ==============================

import json
import torch

# ------------------------------
# 1. SAVE MODEL WEIGHTS
# ------------------------------
torch.save(model.state_dict(), "pure_gnn_model.pth")

print("Model saved as pure_gnn_model.pth")


# ------------------------------
# 2. SAVE FULL MODEL (OPTIONAL)
# ------------------------------
torch.save(model, "pure_gnn_full_model.pth")

print("Full model saved as pure_gnn_full_model.pth")


# ------------------------------
# 3. SAVE PREDICTIONS TO JSON
# ------------------------------
results = []

for i, node in enumerate(nodes):
    results.append({
        "node_id": node["id"],
        "sensor_name": node["name"],
        "sensor_type": node["type"],
        "predicted_label": int(pred[i].cpu().item()),
        "true_label": int(y[i].cpu().item())
    })

with open("gnn_predictions.json", "w") as f:
    json.dump(results, f, indent=4)

print("Predictions saved as gnn_predictions.json")


# ------------------------------
# 4. SAVE GRAPH STRUCTURE + MODEL OUTPUT
# ------------------------------
full_output = {
    "nodes": nodes,
    "edges": edges,
    "predictions": results,
    "final_accuracy": float(accuracy.cpu().item())
}

with open("full_gnn_results.json", "w") as f:
    json.dump(full_output, f, indent=4)

print("Full graph + results saved as full_gnn_results.json")

Model saved as pure_gnn_model.pth
Full model saved as pure_gnn_full_model.pth
Predictions saved as gnn_predictions.json
Full graph + results saved as full_gnn_results.json
